# 63 — CEMS Noncompliance Layer

Consolidates vocabulary hits and table-block hits into a conservative
candidate documentary evidence layer.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def safe_read_parquet(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        df = pd.read_parquet(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def standardise_pollutant(x):
    s = str(x).strip().lower()
    mapping = {
        "oxides of nitrogen": "NOx",
        "nox": "NOx",
        "no2": "NO2",
        "no": "NO",
        "particulates": "Particulates",
        "dust": "Particulates",
        "pm10": "PM10",
        "pm2.5": "PM2.5",
        "sulphur dioxide": "SO2",
        "so2": "SO2",
        "hydrogen chloride": "HCl",
        "hcl": "HCl",
        "total organic carbon": "TOC",
        "toc": "TOC",
        "carbon monoxide": "CO",
        "co": "CO",
        "ammonia": "NH3",
        "nh3": "NH3",
    }
    return mapping.get(s, str(x).strip())

In [ ]:
step = "63_cems_noncompliance_layer"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

vocab, meta_vocab = safe_read_csv(DATA / "cems_vocab_hits.csv")
blocks, meta_blocks = safe_read_csv(DATA / "cems_table_blocks_hits.csv")

frames = []
if not vocab.empty:
    v = vocab.copy()
    v["source_type"] = "vocab_hit"
    frames.append(v)
if not blocks.empty:
    b = blocks.copy()
    b["source_type"] = "table_block_hit"
    frames.append(b)

if not frames:
    raise FileNotFoundError("No CEMS hit files available in data_sources")

combined = pd.concat(frames, ignore_index=True, sort=False)

site_col = next((c for c in ["site_name","site","facility","report_name"] if c in combined.columns), None)
year_col = next((c for c in ["year","report_year"] if c in combined.columns), None)
text_col = next((c for c in ["text","block_text","excerpt","content"] if c in combined.columns), None)

combined["site_name"] = combined[site_col].astype(str).str.strip() if site_col else "unknown"
combined["site_key"] = combined["site_name"].map(slugify)
combined["report_year"] = pd.to_numeric(combined[year_col], errors="coerce") if year_col else np.nan

if text_col:
    txt = combined[text_col].astype(str).str.lower()
    combined["noncompliance_flag"] = txt.str.contains("non-compliance|non compliance|exceed|exceedance|breach|limit", regex=True).astype(int)
else:
    combined["noncompliance_flag"] = 0

summary = (
    combined.groupby(["site_key","site_name","report_year"], as_index=False)
    .agg(
        cems_hit_count=("source_type","size"),
        noncompliance_events=("noncompliance_flag","sum"),
        source_types=("source_type", lambda x: "|".join(sorted(set(map(str, x)))))
    )
    .sort_values(["site_name","report_year"])
)

summary.to_csv(out / "candidate_noncompliance_events.csv", index=False)
write_json(out / "cems_noncompliance_summary.json", {
    "combined_rows": int(len(combined)),
    "site_year_rows": int(len(summary)),
    "sites": int(summary["site_key"].nunique()) if not summary.empty else 0,
})

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml"])
manifest["inputs"] = {"cems_vocab_hits": meta_vocab, "cems_table_blocks_hits": meta_blocks}
add_artifact(manifest, out / "candidate_noncompliance_events.csv")
add_artifact(manifest, out / "cems_noncompliance_summary.json")
write_json(out / "manifest.json", manifest)

display(summary.head(20))